1. Data Loading and Initial Inspection

In [3]:
import pandas as pd

In [ ]:
# Load the triplets dataset
# df = pd.read_csv("triplets_unified.csv", encoding="utf-8")
df = pd.read_csv(r"E:\GraphNLP\triplets_unified.csv", encoding="utf-8")
print(f"Records: {df.shape[0]}, Columns: {list(df.columns)}")
df.head(5)

Records: 633170, Columns: ['head', 'relation', 'tail', 'source', 'pubmed_ids', 'chemical_id', 'disease_id']


C:\Users\hp\AppData\Local\Temp\ipykernel_19372\3243399573.py:3: DtypeWarning: Columns (5,6) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(r"E:\GraphNLP\triplets_unified.csv")


,head,relation,tail,source,pubmed_ids,chemical_id,disease_id
0,10074-G5,associated_with,Adenocarcinoma,chem_dis,26432044,C534883,MESH:D000230
1,10074-G5,associated_with,Adenocarcinoma of Lung,chem_dis,26656844|27602772,C534883,MESH:D000077192
2,10074-G5,associated_with,Alopecia,chem_dis,15902657,C534883,MESH:D000505
3,10074-G5,associated_with,Androgen-Insensitivity Syndrome,chem_dis,1303262|8281139,C534883,MESH:D013734
4,10074-G5,associated_with,Astrocytoma,chem_dis,24680642,C534883,MESH:D001254


2. Normalizing Identifiers to Human-Readable Labels

In [11]:
# load mapping data (this could be from separate CSVs or predefined dicts)
chem_map = {
    "10074-G5": "10,10-bis(4-pyridinylmethyl)-9(10H)-anthracenone",  # example mapping
    # ... (other chemical code->name mappings)
}
disease_map = {
    "MESH:D018567": "Breast Neoplasms, Male",
    "MESH:D013734": "Androgen-Insensitivity Syndrome",
    # ... (other disease ID->name mappings)
}

# Map the chemical and disease identifiers to names
df["chemical_name"] = df["head"].map(chem_map)
df["disease_name"] = df["disease_id"].map(disease_map)

# If some disease names were already in 'tail', use them as fallback
df["disease_name"] = df["disease_name"].fillna(df["tail"])

# Inspect the result for a couple of records
for i in range(3):
    print(df.loc[i, ["head", "chemical_name", "disease_id", "disease_name"]].to_dict())


{'head': '10074-G5', 'chemical_name': '10,10-bis(4-pyridinylmethyl)-9(10H)-anthracenone', 'disease_id': 'MESH:D000230', 'disease_name': 'Adenocarcinoma'}
{'head': '10074-G5', 'chemical_name': '10,10-bis(4-pyridinylmethyl)-9(10H)-anthracenone', 'disease_id': 'MESH:D000077192', 'disease_name': 'Adenocarcinoma of Lung'}
{'head': '10074-G5', 'chemical_name': '10,10-bis(4-pyridinylmethyl)-9(10H)-anthracenone', 'disease_id': 'MESH:D000505', 'disease_name': 'Alopecia'}


3. Refining Labels with NLP (spaCy)

In [12]:
import spacy
nlp = spacy.load("en_core_web_sm")  # load English model

In [13]:
def refine_label(label: str) -> str:
    """Refine a label string for better readability."""
    # Ensure input is a string to avoid TypeErrors
    label = str(label) if not pd.isna(label) else ""

    # Rule 1: If label contains a comma, attempt to reorder parts
    if "," in label:
        parts = [p.strip() for p in label.split(",")]
        if len(parts) == 2:
            # Identify part-of-speech of the second part
            doc = nlp(parts[1])
            pos = doc[0].pos_ if len(doc) > 0 else ""
            # If second part is an adjective or noun, swap order
            if pos in {"ADJ", "NOUN"}:
                label = f"{parts[1]} {parts[0]}"

    # Rule 2: Replace certain formal terms with lay terms
    if label.endswith("Neoplasms"):
        label = label.replace("Neoplasms", "Cancer")

    return label

In [14]:
# Apply refinement to disease names
df["disease_name_refined"] = df["disease_name"].apply(refine_label)

# Example transformation:
print(refine_label("Breast Neoplasms, Male"))  # -> "Male Breast Cancer"


Male Breast Cancer


In [15]:
# Apply the same refinement to chemical names if they contain commas or complex terms
df["chemical_name_refined"] = df["chemical_name"].apply(refine_label)

# For our dataset, chemical names like "10,10-bis(4-pyridinylmethyl)-9(10H)-anthracenone" 
# don't need reordering, so this step may not change much.


Synonym Handling

In [16]:
# 3.1 Standardizing synonyms using dictionary
synonym_dict = {
    "Hair Loss": "Alopecia",
    "High Blood Sugar": "Hyperglycemia",
    "Male Pattern Baldness": "Androgenetic Alopecia"
    # Add more known synonyms
}

# Apply to disease and chemical names
df["disease_name_refined"] = df["disease_name_refined"].replace(synonym_dict)
df["chemical_name_refined"] = df["chemical_name_refined"].replace(synonym_dict)


Ontology-Based Entity Linking

In [30]:
!pip install scispacy==0.5.4


     ---------------------------------------- 0.0/42.1 MB ? eta -:--:--
     ---------------------------------------- 0.0/42.1 MB ? eta -:--:--
     ---------------------------------------- 0.0/42.1 MB ? eta -:--:--
     ---------------------------------------- 0.3/42.1 MB ? eta -:--:--
     ---------------------------------------- 0.3/42.1 MB ? eta -:--:--
     --------------------------------------- 0.5/42.1 MB 578.7 kB/s eta 0:01:12
     --------------------------------------- 0.5/42.1 MB 578.7 kB/s eta 0:01:12
      -------------------------------------- 0.8/42.1 MB 599.2 kB/s eta 0:01:09
      -------------------------------------- 1.0/42.1 MB 680.3 kB/s eta 0:01:01
     - ------------------------------------- 1.3/42.1 MB 737.4 kB/s eta 0:00:56
     - ------------------------------------- 1.3/42.1 MB 737.4 kB/s eta 0:00:56
     - ------------------------------------- 1.6/42.1 MB 791.5 kB/s eta 0:00:52
     - ------------------------------------- 1.8/42.1 MB 792.8 kB/s eta 0:00:51


  error: subprocess-exited-with-error
  
  Preparing metadata (pyproject.toml) did not run successfully.
  exit code: 1
  
  [45 lines of output]
  + meson setup C:\Users\hp\AppData\Local\Temp\pip-install-l3r9nf_g\scipy_45216af28b33452bb729e4697f58e233 C:\Users\hp\AppData\Local\Temp\pip-install-l3r9nf_g\scipy_45216af28b33452bb729e4697f58e233\.mesonpy-uc1_gc63 -Dbuildtype=release -Db_ndebug=if-release -Db_vscrt=md --native-file=C:\Users\hp\AppData\Local\Temp\pip-install-l3r9nf_g\scipy_45216af28b33452bb729e4697f58e233\.mesonpy-uc1_gc63\meson-python-native-file.ini
  The Meson build system
  Version: 1.8.2
  Source dir: C:\Users\hp\AppData\Local\Temp\pip-install-l3r9nf_g\scipy_45216af28b33452bb729e4697f58e233
  Build dir: C:\Users\hp\AppData\Local\Temp\pip-install-l3r9nf_g\scipy_45216af28b33452bb729e4697f58e233\.mesonpy-uc1_gc63
  Build type: native build
  Activating VS 17.14.9 (July 2025)
  Project name: SciPy
  Project version: 1.9.3
  C compiler for the host machine: cl (msvc 19.44.35

In [ ]:
!pip install https://huggingface.co/allenai/scispacy_models/resolve/main/en_core_sci_sm-0.5.4.tar.gz


In [26]:
!E:\GraphNLP\.venv\Scripts\python.exe -m pip install scispacy

  Using cached scispacy-0.5.5-py3-none-any.whl (46 kB)
  Using cached nmslib-2.1.1.tar.gz (188 kB)
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Using cached spacy-3.7.5-cp310-cp310-win_amd64.whl (12.1 MB)
  Using cached thinc-8.2.5-cp310-cp310-win_amd64.whl (1.5 MB)
  Using cached weasel-0.4.1-py3-none-any.whl (50 kB)
  Running setup.py install for nmslib: started
  Running setup.py install for nmslib: finished with status 'error'


  DEPRECATION: nmslib is being installed using the legacy 'setup.py install' method, because it does not have a 'pyproject.toml' and the 'wheel' package is not installed. pip 23.1 will enforce this behaviour change. A possible replacement is to enable the '--use-pep517' option. Discussion can be found at https://github.com/pypa/pip/issues/8559
  error: subprocess-exited-with-error
  
  Running setup.py install for nmslib did not run successfully.
  exit code: 1
  
  [1145 lines of output]
  Dependence list: ['pybind11<2.6.2', 'psutil', "numpy>=1.10.0,<1.17 ; python_version=='2.7'", "numpy>=1.10.0 ; python_version>='3.5'"]
  E:\GraphNLP\.venv\lib\site-packages\setuptools\dist.py:771: UserWarning: Usage of dash-separated 'description-file' will not be supported in future versions. Please use the underscore name 'description_file' instead
    warnings.warn(
  running install
  E:\GraphNLP\.venv\lib\site-packages\setuptools\command\install.py:34: SetuptoolsDeprecationWarning: setup.py inst

In [29]:
!E:\GraphNLP\.venv\Scripts\python.exe -m pip install https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/en_core_sci_sm-0.5.1.tar.gz

  ERROR: HTTP error 404 while getting https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/en_core_sci_sm-0.5.1.tar.gz
ERROR: Could not install requirement https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/en_core_sci_sm-0.5.1.tar.gz because of HTTP error 404 Client Error: Not Found for url: https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/en_core_sci_sm-0.5.1.tar.gz for URL https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/en_core_sci_sm-0.5.1.tar.gz

[notice] A new release of pip is available: 23.0.1 -> 25.1.1
[notice] To update, run: E:\GraphNLP\.venv\Scripts\python.exe -m pip install --upgrade pip


In [27]:
from scispacy.linking import EntityLinker

ModuleNotFoundError: No module named 'scispacy'

In [ ]:
# 3.2 Link to ontology using SciSpaCy


nlp_linker = spacy.load("en_core_sci_sm")
nlp_linker.add_pipe("scispacy_linker", config={"resolve_abbreviations": True, "name": "umls"})

# Function to get top concept from UMLS
def link_entity(name):
    doc = nlp_linker(name)
    for ent in doc.ents:
        for umls_id, score in ent._.kb_ents:
            return umls_id  # You may also extract canonical name
    return None

# Link diseases
df["umls_id_disease"] = df["disease_name_refined"].apply(link_entity)
# also link chemicals
# df["umls_id_chemical"] = df["chemical_name_refined"].apply(link_entity)


ModuleNotFoundError: No module named 'scispacy'

4. Constructing the Final Triplet Table

In [ ]:
# Select and rename columns for the final triplet table
final_df = df[["chemical_name_refined", "relation", "disease_name_refined"]].copy()
final_df.columns = ["Chemical", "Relation", "Disease"]

# Drop duplicates if any overlapping triplets exist
final_df = final_df.drop_duplicates().reset_index(drop=True)

print(final_df.head(5))
print("Total triplets:", len(final_df))


                                           Chemical         Relation  \
0  10,10-bis(4-pyridinylmethyl)-9(10H)-anthracenone  associated_with   
1  10,10-bis(4-pyridinylmethyl)-9(10H)-anthracenone  associated_with   
2  10,10-bis(4-pyridinylmethyl)-9(10H)-anthracenone  associated_with   
3  10,10-bis(4-pyridinylmethyl)-9(10H)-anthracenone  associated_with   
4  10,10-bis(4-pyridinylmethyl)-9(10H)-anthracenone  associated_with   

                           Disease  
0                   Adenocarcinoma  
1           Adenocarcinoma of Lung  
2                         Alopecia  
3  Androgen-Insensitivity Syndrome  
4                      Astrocytoma  
Total triplets: 20730


4.1. Enrichment (descriptions, IUPAC names, etc

In [ ]:
# 4.1 Enrich diseases with descriptions
mesh_definitions = {
    "Alopecia": "Loss of hair from the head or body, sometimes as a result of heredity, illness, or a drug."
    # Build full map from MeSH or BioPortal
}
final_df["description"] = final_df["Disease"].map(mesh_definitions)

# enrich chemicals with PubChem
# import pubchempy as pcp
# def get_iupac(name):
#     try:
#         return pcp.get_compounds(name, "name")[0].iupac_name
#     except: return None
# final_df["iupac_name"] = final_df["Chemical"].apply(get_iupac)


5. Preparing Data for Neo4j Import

5.1: Create Nodes CSVs

In [ ]:
# Prepare chemical nodes dataframe
chem_nodes = df[["chemical_id", "chemical_name_refined"]].drop_duplicates().copy()
chem_nodes.columns = ["chem_id", "name"]   # rename for clarity

# Prepare disease nodes dataframe
disease_nodes = df[["disease_id", "disease_name_refined"]].drop_duplicates().copy()
disease_nodes.columns = ["disease_id", "name"]

# Save nodes to CSV
chem_nodes.to_csv("chem_nodes.csv", index=False)
disease_nodes.to_csv("disease_nodes.csv", index=False)
print(f"Saved {len(chem_nodes)} chemical nodes and {len(disease_nodes)} disease nodes.")

Saved 3690 chemical nodes and 13243 disease nodes.


5.2: Create Relationships CSV

In [ ]:
# Prepare relationships dataframe
rel_df = df[["chemical_id", "disease_id", "relation"]].copy()
rel_df.columns = ["chem_id", "disease_id", "relation_type"]

# If we standardized relation to uppercase in final_df, do the same here
rel_df["relation_type"] = rel_df["relation_type"].str.upper().str.replace(" ", "_")

rel_df.to_csv("relationships.csv", index=False)
print(f"Saved {len(rel_df)} relationships.")


Saved 633170 relationships.
